# Identifier-masked probe — the reviewer's first ask

**Runtime → GPU. Run all.** About 90 minutes.

The probe pools activations over the occurrence's own tokens, so it reads the
variable's *name*. The surface baseline is forbidden from reading it. That
asymmetry is disclosed in the paper but not controlled, and it is the single
biggest threat to the claim that the probe encodes roles rather than names.

This notebook removes it. `--pool context` averages the tokens **around** the
occurrence and excludes its own, so the forward pass is byte-identical to the
span-pooled run and only the read changes. Masking the name in the source would
also change the input distribution; this does not.

It produces the four-way comparison the review asks for, on the same
occurrences:

| | reads the name? | reads context? |
|---|---|---|
| probe, span-pooled | yes | no |
| **probe, context-pooled** | **no** | **yes** |
| surface n-gram, masked | no | yes |
| identifier alone | yes | no |

It also runs on the **three-way intersection** (869 problems shared by all
three languages) so every cell scores identical problem ids, and emits per-cell
predictions so the probe side gets problem-level confidence intervals.

In [ ]:
# 1 - setup
import pathlib, os
BRANCH = "main"
REPO = "/content/mech-interp"
if not pathlib.Path(REPO).exists():
    !git clone -q https://github.com/nolanlwin/mech-interp.git {REPO}
%cd {REPO}
!git fetch -q origin && git checkout -q -B {BRANCH} origin/{BRANCH} && git pull -q
!git log --oneline -1
!pip install -q transformers==5.8.0 torch numpy scikit-learn matplotlib tree_sitter \
  "tree-sitter-javascript>=0.25.0" "tree-sitter-php>=0.24.1"
try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except Exception:
    pass
from google.colab import drive
drive.mount("/content/drive")
DEST = "/content/drive/MyDrive/mech-interp/masked"
!mkdir -p data/xlcost outputs/role_occ outputs/activations_xlcost outputs/crosslang {DEST}
!cp -rn {DEST}/stores/* outputs/activations_xlcost/ 2>/dev/null || true
!cp -n  {DEST}/role_occ/* outputs/role_occ/ 2>/dev/null || true
!cp -n  {DEST}/data_xlcost/* data/xlcost/ 2>/dev/null || true

REQUIRED = ["scripts/extract_activations.py", "scripts/crosslang.py",
            "scripts/baselines.py", "scripts/role_occurrences.py",
            "scripts/build_intersection.py", "scripts/transfer_intervals.py"]
missing = [f for f in REQUIRED if not pathlib.Path(f).exists()]
if missing:
    raise SystemExit(f"BRANCH={BRANCH!r} lacks {missing}; point BRANCH at the branch that has them.")
import torch
print(f"setup complete | cuda={torch.cuda.is_available()} "
      f"{torch.cuda.get_device_name(0) if torch.cuda.is_available() else ''}")

In [ ]:
# 2 - CONFIG
MODEL  = "Qwen/Qwen2.5-Coder-1.5B"
ROLES  = ["accumulator", "iterator", "index_key"]
LANGS  = {"Python": "python", "Javascript": "javascript", "PHP": "php"}
SPLIT  = "train"
CONTEXT_TOKENS = 16          # tokens either side when pooling context only
PRIMARY = "window_masked"    # the pre-registered surface variant, not a per-cell max

import re as _re
slug = lambda mid: _re.sub(r"[^a-z0-9]", "", mid.split("/")[-1].lower())
SPAN_SLUG = slug(MODEL)
CTX_SLUG  = slug(f"{MODEL}#pool-context{CONTEXT_TOKENS}")
print(f"{MODEL}\n  span-pooled store slug   : {SPAN_SLUG}\n  context-pooled store slug: {CTX_SLUG}")

In [ ]:
# 3 - corpora, occurrences, and the three-way intersection. CPU, ~15 min.
import json, itertools
for L, s in LANGS.items():
    if not pathlib.Path(f"data/xlcost/{s}_{SPLIT}.jsonl.stats.json").exists():
        !python scripts/xlcost_data.py build --language "{L}" --split {SPLIT} --out-dir data/xlcost
for L, s in LANGS.items():
    occ = f"outputs/role_occ/all_{s}_{SPLIT}.jsonl"
    if not pathlib.Path(occ + ".stats.json").exists():
        !python scripts/role_occurrences.py extract --input data/xlcost/{s}_{SPLIT}.jsonl --role all --output {occ}

!python scripts/build_intersection.py

absent = [f"outputs/role_occ/isect_{s}_{SPLIT}.jsonl" for s in LANGS.values()
          if not pathlib.Path(f"outputs/role_occ/isect_{s}_{SPLIT}.jsonl").exists()]
if absent:
    raise SystemExit(f"intersection build produced nothing for {absent}; read the output above.")
for s in LANGS.values():
    n = sum(1 for _ in open(f"outputs/role_occ/isect_{s}_{SPLIT}.jsonl"))
    print(f"  {s}: {n} occurrences on the shared problem set")

In [ ]:
# 4 - activations. GPU. Two stores per language: span-pooled and context-pooled.
#     ~90 min total. The context store is the identifier-masked condition.
for L, s in LANGS.items():
    for pool, sl in (("span", SPAN_SLUG), ("context", CTX_SLUG)):
        out = f"outputs/activations_xlcost/isect_{s}_{SPLIT}_{sl}"
        if pathlib.Path(f"{out}/meta.json").exists():
            print(f"  have {out}")
            continue
        extra = f"--pool context --context-tokens {CONTEXT_TOKENS}" if pool == "context" else ""
        !python scripts/extract_activations.py run \
          --canonical data/xlcost/{s}_{SPLIT}_isect.jsonl \
          --occurrences outputs/role_occ/isect_{s}_{SPLIT}.jsonl \
          --model-id {MODEL} --label-field role {extra} --out-dir {out}

bad = [f"outputs/activations_xlcost/isect_{s}_{SPLIT}_{sl}"
       for s in LANGS.values() for sl in (SPAN_SLUG, CTX_SLUG)
       if not pathlib.Path(f"outputs/activations_xlcost/isect_{s}_{SPLIT}_{sl}/meta.json").exists()]
if bad:
    raise SystemExit(f"no store written for {bad}; cell 5 has nothing to probe.")
print("\nall six stores present")

In [ ]:
# 5 - probe transfer, both poolings, all six ordered pairs. GPU-light.
import itertools, glob
for sl, tag in ((SPAN_SLUG, "span"), (CTX_SLUG, "context")):
    for role in ROLES:
        for a, b in itertools.permutations(LANGS.values(), 2):
            out = f"outputs/crosslang/probe_{role}_{a}_to_{b}_{sl}.json"
            if pathlib.Path(out).exists():
                continue
            !python scripts/crosslang.py run \
              --train-store outputs/activations_xlcost/isect_{a}_{SPLIT}_{sl} \
              --test-store  outputs/activations_xlcost/isect_{b}_{SPLIT}_{sl} \
              --role {role} --output {out}
made = glob.glob("outputs/crosslang/probe_*.json")
if not made:
    raise SystemExit("no probe results written; read the output above.")
print(f"{len(made)} probe result(s)")

In [ ]:
# 6 - the model-free arm on the SAME occurrences, one pre-registered variant.
#     CPU, ~20 min. --primary-feature pins which family the emitted predictions
#     belong to, so the intervals in cell 7 describe one named estimator.
for role in ROLES:
    for a, b in itertools.permutations(LANGS.values(), 2):
        out = f"outputs/isect_wm/{role}_{a}_to_{b}.json"
        if pathlib.Path(out).exists():
            continue
        !python scripts/baselines.py transfer \
          --train-occurrences outputs/role_occ/isect_{a}_{SPLIT}.jsonl \
          --train-canonical   data/xlcost/{a}_{SPLIT}_isect.jsonl \
          --test-occurrences  outputs/role_occ/isect_{b}_{SPLIT}.jsonl \
          --test-canonical    data/xlcost/{b}_{SPLIT}_isect.jsonl \
          --label-field role --role {role} --matched \
          --primary-feature {PRIMARY} --output {out}
print(f"{len(glob.glob('outputs/isect_wm/*.json'))} baseline cell(s)")

In [ ]:
# 7 - problem-level confidence intervals, and the four-way comparison.
!python scripts/transfer_intervals.py --in outputs/isect_wm \
    --out results/lp4fm/transfer_intervals.csv --n-boot 2000

import csv, statistics as st, json
f = lambda r, k: float(r[k])
near = lambda rs: [r for r in rs if "python" not in (r["source"], r["target"])]
far  = lambda rs: [r for r in rs if "python" in (r["source"], r["target"])]
m    = lambda g, k: st.mean(f(r, k) for r in g)

def probe_rows(sl):
    out = []
    for p in glob.glob(f"outputs/crosslang/probe_*_{sl}.json"):
        d = json.loads(open(p).read())
        mm = _re.match(r"probe_([a-z_]+)_([a-z]+)_to_([a-z]+)_", pathlib.Path(p).name)
        if mm:
            out.append({"role": mm.group(1), "source": mm.group(2), "target": mm.group(3),
                        "v": d["transfer_macro_f1_mean"]})
    return out

print(f"\n{'condition':<34}{'close':>9}{'Python':>9}{'effect':>9}")
base = list(csv.DictReader(open("results/lp4fm/transfer_intervals.csv")))
base = [r for r in base if r["role"] != "ALL"]
print(f"{'surface n-gram (' + PRIMARY + ')':<34}{m(near(base),'macro_f1'):>9.3f}"
      f"{m(far(base),'macro_f1'):>9.3f}{m(far(base),'macro_f1')-m(near(base),'macro_f1'):>+9.3f}")
for sl, tag in ((SPAN_SLUG, "probe, span-pooled (reads name)"),
                (CTX_SLUG,  "probe, context-pooled (no name)")):
    rs = probe_rows(sl)
    if rs:
        print(f"{tag:<34}{m(near(rs),'v'):>9.3f}{m(far(rs),'v'):>9.3f}"
              f"{m(far(rs),'v')-m(near(rs),'v'):>+9.3f}")

print("\nIf the context-pooled probe stays flat, the invariance is not carried by")
print("the identifier. If it collapses toward the surface curve, the paper's")
print("central claim needs restating and the abstract must say so.")

In [ ]:
# 8 - save to Drive. Push results only; nothing here is anonymised.
!mkdir -p {DEST}/stores {DEST}/role_occ {DEST}/data_xlcost {DEST}/crosslang {DEST}/isect_wm
!cp -r outputs/activations_xlcost/* {DEST}/stores/ 2>/dev/null || true
!cp outputs/role_occ/* {DEST}/role_occ/ 2>/dev/null || true
!cp data/xlcost/*_{SPLIT}*.jsonl* {DEST}/data_xlcost/ 2>/dev/null || true
!cp outputs/crosslang/*.json {DEST}/crosslang/ 2>/dev/null || true
!cp outputs/isect_wm/*.json {DEST}/isect_wm/ 2>/dev/null || true
print(f"artifacts saved to {DEST}")
print("\nPaste cell 7's table back into the chat; the paper's numbers follow from it.")